# Entwicklungs-EDA: Alter, Zusammensetzung und Verlustkosten

Dieses Notebook prüft die gespeicherten Ergebnisse von `python -m src.eda`. Es analysiert keine Evaluationszeilen, passt kein Modell an und verändert keine Vertragszuordnung. [Fragen, fünf Befunde und Grenzen](../docs/eda.md) erläutern die Ergebnisse. Die frühere vollständige Datenexploration bleibt offengelegt; dies ist keine externe Validierung.

In [ ]:
from pathlib import Path
import sys
import json

start = Path.cwd().resolve()
ROOT = next(p for p in [start, *start.parents] if (p / 'config/eda_protocol.json').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.eda import load_report
from IPython.display import display, Image

tables, report = load_report()
assert report['evaluation_rows_analyzed'] == 0 and report['models_fitted'] == 0
source = json.loads((ROOT / 'reports/s1/preparation_report.json').read_text(encoding='utf-8'))
split = json.loads((ROOT / 'reports/s2/split_manifest.json').read_text(encoding='utf-8'))
assert report['identity']['source_id'] == source['source_id']
assert report['identity']['preparation_id'] == source['preparation_id']
assert report['identity']['split_id'] == split['split_id']
print('Lauf:', report['run_id'])
print('Quelle:', report['identity']['source_id'], '| Aufteilung:', report['identity']['split_id'])
print('Alle lokalen Tabellen, öffentlichen Spiegel, Bild- und Code-Hashes geprüft.')

## Beobachtete Größen und Unterstützung

Die Frequenz ist Anzahl durch Vertragsjahre, das Schadenmittel Betrag durch Anzahl, der jährliche Verlust Betrag durch Vertragsjahre. Das sind beobachtete Größen einschließlich Nullverträgen, keine vorhergesagten Tarife. Bei positiven Anzahlen gilt die beobachtete Produktidentität. Häufigkeiten sind keine Schadenwahrscheinlichkeitsprozente.

In [ ]:
GERMAN = {'age_band': 'DrivAge-Gruppe', 'age10_band': 'DrivAge-Gruppe', 'policies': 'Verträge', 'exposure': 'Vertragsjahre', 'claims': 'Schäden', 'loss_eur': 'Betrag EUR', 'frequency': 'Schäden/Jahr', 'severity_eur': 'EUR/Schaden', 'loss_cost_eur_per_year': 'EUR/Jahr', 'supported': 'Mindestens 500 Jahre', 'standardized_frequency': 'BM-standardisierte Frequenz', 'lower': 'Intervall unten', 'upper': 'Intervall oben', 'excluded_policies': 'Ausgeschlossene Verträge', 'excluded_exposure': 'Ausgeschlossene Jahre', 'excluded_claims': 'Ausgeschlossene Schäden', 'excluded_loss_eur': 'Ausgeschlossene EUR', 'scenario': 'Szenario', 'removed_policies': 'Entfernte Verträge', 'removed_exposure': 'Entfernte Jahre', 'removed_claims': 'Entfernte Schäden', 'removed_loss_eur': 'Entfernte EUR', 'top_claim_rows': 'Größte Schadenzeilen', 'claim_row_share': 'Anteil Schadenzeilen', 'loss_share': 'Anteil Betrag', 'feature': 'Merkmal', 'level': 'Ausprägung', 'exposure_share_within_age': 'Exponierungsanteil im Alter', 'frequency_change_pct': 'Frequenzänderung in Prozent', 'loss_cost_change_pct': 'Verlustkostenänderung in Prozent'}
def show(frame):
    display(frame.rename(columns=GERMAN))
show(tables['age_band'][['age_band', 'policies', 'exposure', 'claims', 'loss_eur', 'frequency', 'severity_eur', 'loss_cost_eur_per_year', 'supported']])
print('Abweichende mitgelieferte/rekonstruierte Anzahlen:', report['supplied_reconstructed_count_differences'])
print('Erhaltene exakt wiederholte Vertrags-/Betragszeilen:', report['exact_repeated_policy_amount_rows_retained'])

In [ ]:
display(Image(filename=str(ROOT / 'reports/s3/figures/alter_beobachtet.png')))
display(Image(filename=str(ROOT / 'reports/s3/figures/alter_robustheit.png')))

## Alter und Bonus-Malus: beschränkte deskriptive Standardisierung

Die Frequenz zeigt nach einem Rückgang einen breiten Wiederanstieg im mittleren Alter. Benachbarte Fünfjahreskontraste bleiben unklar. Die BM-Standardisierung verwendet gemeinsame Exponierungsgewichte und nur in jeder Vergleichsgruppe unterstützte BM-Zellen. Ausgeschlossene Unterstützung wird gezeigt, im Primärbestand aber erhalten. Dies ist keine vollständige oder kausale Adjustierung. Fahrzeug-/Kraftstoff-/Regionszusammensetzung bleibt als alternative Erklärung offen.

In [ ]:
show(tables['bm_standardized'][['age10_band', 'policies', 'exposure', 'claims', 'frequency', 'standardized_frequency', 'lower', 'upper', 'excluded_policies', 'excluded_exposure', 'excluded_claims', 'excluded_loss_eur']])
display(Image(filename=str(ROOT / 'reports/s3/figures/alter_zusammensetzung.png')))
show(tables['age_composition'].query("feature in ['VehGas', 'VehBrand']")[['age_band', 'feature', 'level', 'exposure', 'claims', 'frequency', 'exposure_share_within_age', 'supported']])

## Portfoliomerkmale, kurze Perioden und B12

Markenklassen sind anonymisiert; B12 ist kein bestätigter Fehler. Eine kurze Beobachtungsperiode kann mit anderem Risikomix und unbeobachteten Beendigungsprozessen verbunden sein. Hohe Randraten beweisen keinen Offsetfehler. Poisson-Intervalle in den Merkmalsprofilen sind punktweise Referenzintervalle unter der Poisson-Annahme; sie berücksichtigen keine unbekannte Überdispersion/Abhängigkeit.

In [ ]:
display(Image(filename=str(ROOT / 'reports/s3/figures/portfolio_raten.png')))
show(tables['Region'][['Region', 'policies', 'exposure', 'claims', 'loss_eur', 'frequency', 'supported']])
show(tables['b12_new'][['brand12', 'brand_new', 'policies', 'exposure', 'claims', 'loss_eur', 'frequency']])
show(tables['exposure_band'][['exposure_band', 'policies', 'exposure', 'claims', 'loss_eur', 'frequency', 'supported']])

## Getrennte Sensitivitäten und oberer Schadenrand

Ausschlüsse ändern die Population; Kappungen von Exponierung, Anzahl oder Betrag ändern Nenner/Ziel. Szenarien sind einzeln gegenüber derselben ungekappter Basis berechnet und nicht addierbar. Ein Anzahlcap ohne passende Schadenzeilen darf kein konsistentes Schadenmittel vortäuschen. Null ist etwas anderes als undefiniert. Die 500 Vertragsbootstrap-Ziehungen halten Anzahl, Exponierung und Betrag zusammen, erfassen aber keine unbeobachteten Extremverluste oder Haushaltsabhängigkeit.

In [ ]:
show(tables['sensitivities'][['scenario', 'policies', 'exposure', 'claims', 'loss_eur', 'severity_eur', 'removed_policies', 'removed_exposure', 'removed_claims', 'removed_loss_eur', 'frequency_change_pct', 'loss_cost_change_pct']])
show(tables['tail_concentration'])
display(Image(filename=str(ROOT / 'reports/s3/figures/alter_sensitivitaet.png')))
display(Image(filename=str(ROOT / 'reports/s3/figures/pruefgruppen_verluste.png')))

## Was folgt – und was offen bleibt

GLM und GAM bleiben substanzielle Kandidaten neben dem Histogramm-Poisson-Vergleich. Flexible Alters-/BM-Terme, gemeinsame Kompositionsdiagnostik und Exponierungsdiagnostik sind begründet; eine Interaktion ist noch nicht gewählt. Die gemessene Bestandszusammensetzung legt eine gemeinsame Modellierung von Alter und Bonus-Malus nahe. Zusätzliche Unterschiede in Fahrleistung und Nutzung bleiben mögliche, hier nicht gemessene Erklärungen. Erst die späteren Modelle liefern adjustierte Vorhersageprofile und eine unabhängig geprüfte Frequenz-Schadenhöhenzerlegung. [Vollständige Evidenz und Primärquellen](../docs/eda.md).